# **Cultura e Práticas em DataOps e MLOps**
**Autor**: Renan Santos Mendes

**Email**: renansantosmendes@gmail.com

**Descrição**: Este notebook apresenta um exemplo de como consumir um modelo registrado no MLflow Model Registry, realizando o carregamento e a inferência com o modelo treinado.

**OBSERVAÇÃO**: Este notebook representa uma versão atualizada do fluxo de trabalho de MLOps, cujas modificações foram necessárias em decorrência de atualizações realizadas na própria biblioteca do **MLflow**. Algumas interfaces e métodos utilizados em versões anteriores sofreram alterações ou foram descontinuados, exigindo a adoção de novas abordagens para garantir a compatibilidade e o correto funcionamento do pipeline. Um exemplo direto dessas mudanças é a forma de **leitura e carregamento de modelos registrados**, que passou a exigir o uso do **`MlflowClient`** e da interface **`mlflow.pyfunc`** para acessar e consumir modelos armazenados no Model Registry, substituindo métodos que não são mais suportados nas versões mais recentes da biblioteca.
"""

In [ ]:
%%capture
!uv pip install mlflow -q

# Instalando pacotes

In [ ]:
import os
import mlflow
import numpy as np

### **Importação de Pacotes Necessários**

Nesta etapa, são importados os pacotes necessários para a execução do projeto.

- O `mlflow` é utilizado para interagir com o servidor de rastreamento e o registro de modelos, permitindo o carregamento de modelos registrados de forma padronizada.
- O `os` é utilizado para configurar variáveis de ambiente, como credenciais de autenticação e a URI do servidor de rastreamento.
- O `numpy` é utilizado para manipulação dos dados de entrada que serão enviados ao modelo para inferência.

Essa preparação é essencial para garantir a comunicação adequada com o servidor MLflow e viabilizar o consumo do modelo registrado.

In [ ]:
os.environ['MLFLOW_TRACKING_USERNAME'] = 'renansantosmendes' # ALTERE PARA O SEU PRÓPRIO USERNAME
os.environ['MLFLOW_TRACKING_PASSWORD'] = 'cc41cc48f8e489dd5b87404dd6f9720944e32e9b' # VALIDEM SE O TOKEN ESTÁ ATIVO, SE NÃO ESTIVER CRIE O SEU PRÓPRIO REPOSITÓRIO E SEU TOKEN
mlflow.set_tracking_uri('https://dagshub.com/renansantosmendes/mlops-ead-registry.mlflow') # EXEMPLO: https://dagshub.com/myuser/myrepo.mlflow'

### **Configurando o MLflow**

Esta etapa configura o MLflow para se comunicar com um servidor remoto de rastreamento hospedado no DagsHub. As credenciais de autenticação são fornecidas por meio de variáveis de ambiente, garantindo o acesso seguro ao serviço.

Em seguida, a URI do servidor de rastreamento é definida, estabelecendo o servidor remoto do MLflow como destino padrão para todas as operações de leitura e registro de experimentos realizadas neste notebook.

Essa configuração é fundamental para que o cliente MLflow consiga localizar e acessar os modelos registrados no ambiente remoto.

In [ ]:
client = mlflow.MlflowClient(tracking_uri=mlflow.get_tracking_uri())

In [ ]:
client

### **Criando um Client para Comunicação com o Registro no DagsHub**

O `MlflowClient` é a interface programática do MLflow para interagir diretamente com o servidor de rastreamento e o registro de modelos. Ao instanciá-lo com a URI de rastreamento configurada anteriormente, é estabelecida uma conexão com o servidor remoto no DagsHub.

Por meio desse cliente, é possível consultar experimentos, execuções, modelos registrados e suas versões, bem como realizar operações administrativas no ciclo de vida dos modelos, como transição de estágios e adição de tags.

In [ ]:
registered_model = client.get_registered_model('fetal_health')

In [ ]:
registered_model.latest_versions

### **Recebendo o Modelo Registrado e Suas Versões**

Nesta etapa, o modelo registrado no MLflow Model Registry é recuperado pelo nome `fetal_health`, que corresponde ao modelo treinado para classificação da saúde fetal. O método `get_registered_model` retorna um objeto contendo metadados do modelo, incluindo suas versões disponíveis.

A partir do atributo `latest_versions`, é possível inspecionar todas as versões mais recentes do modelo por estágio (por exemplo, `Staging`, `Production` ou `None`), permitindo selecionar a versão mais adequada para carregamento e inferência.

In [ ]:
model_uri = registered_model.latest_versions[-1].source

### **Obtendo o URI da Execução do Modelo**

Nesta etapa, o URI de origem da versão mais recente do modelo registrado é extraído. Esse URI aponta para o artefato do modelo armazenado no servidor MLflow, identificando o caminho exato de onde o modelo treinado pode ser carregado.

A última versão da lista `latest_versions` é selecionada, e o atributo `source` fornece o endereço do artefato correspondente, que será utilizado na etapa de carregamento do modelo.

In [ ]:
loaded_model = mlflow.pyfunc.load_model(model_uri)

### **Carregando o Modelo**

O modelo é carregado a partir do URI obtido anteriormente utilizando a interface `mlflow.pyfunc`, que oferece uma abstração genérica para modelos registrados no MLflow, independentemente do framework utilizado em seu treinamento.

Ao carregar o modelo como `pyfunc`, garante-se uma interface padronizada de inferência por meio do método `predict`, permitindo que o modelo seja consumido de forma consistente em diferentes contextos de produção.

In [ ]:
accelerations = 0
fetal_movement = 0
uterine_contractions = 0
severe_decelerations = 0

In [ ]:
received_data = np.array([
        accelerations,
        fetal_movement,
        uterine_contractions,
        severe_decelerations,
    ], np.float32).reshape(1, -1)

In [ ]:
received_data

In [ ]:
loaded_model.predict(received_data)